# The Kakeya conjecture

*maths · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/kakeya_conjecture)

A needle can be turned through every direction inside a set of arbitrarily small area. This problem builds such a set (Perron's tree of $2^k$ thin triangles, shifted sideways) and measures its area exactly, row by row. It then covers unions of $\delta$-tubes in $\mathbb R^2$, $\mathbb R^3$ and $\mathbb R^4$ with $\delta$-cubes to estimate their dimension. The Kakeya conjecture says such sets still have full dimension $n$. That is proven for $n = 2$ and, since 2025, $n = 3$; for $n \ge 4$ the only proofs so far are unverified, AI-generated claims.

**Vector form**

$$|T_1 \cup \dots \cup T_N| \le \big(\alpha^{2k} + 2(1-\alpha)\big)\,|T| \xrightarrow[k\to\infty]{} 0,\qquad \dim_M K = \lim_{\delta\to0}\frac{\log N_\delta(K)}{\log(1/\delta)} \overset{?}{=} n \;\text{ for every Kakeya set } K\subset\mathbb R^n$$

**Index (tensor) form**

$$\ell_{yi} = b_i(1 - y) + s_i,\quad r_{yi} = \ell_{yi} + w_0(1-y),\qquad |\textstyle\bigcup T| = \frac1Y\sum_y \sum_i \big(r_{y(i)} - \max(\ell_{y(i)}, \max_{j<i} r_{y(j)})\big)_+,\qquad N_\delta = \#\big\{\lfloor (c_{j} + t\,u_{j})/\delta \rfloor\big\}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $b_i,\ w_0$ | `triangle_bases(k, device)` | (N,), float | the $N = 2^k$ thin triangles with apex $(0, 1)$ and bases $[b_i, b_i + w_0]$ splitting $[-1/\sqrt3, 1/\sqrt3]$: a 60° cone of directions, every side $\ge 1$ |
| $s_i$ | `perron_shifts(k, alphas, h)` | (A, N) | horizontal shifts after $k$ Perron rounds, for $A$ values of $\alpha$ at once; each round makes neighbouring hearts touch and pushes the right one left by $(1-\alpha)(c - a)$ |
| $\ell_{yi},\ r_{yi}$ | `intervals(y, base, w0, shift)` | (A, Y, N) | where triangle $i$ cuts the horizontal line at height $y$ |
| $\max_{j<i} r_{y(j)}$ | `cummax(gather(hi, -1, argsort(lo)), dim=-1)` | (A, Y, N) | sort the intervals of each row by their left end; the running maximum of the right ends gives what is already covered |
| $\vert \bigcup T\vert $ | `union_area(base, w0, shift, n_rows)` | (A,) | repeated $i$ and $y$ ⇒ summed: the exact union length of every row, averaged over rows (midpoint rule) |
| $u_j$ | `directions(n, delta, device)` | (D, n) | a $\delta$-grid on the faces of the cube $[-1,1]^n$, pushed to the sphere: about $\delta$-separated line directions |
| $\lfloor (c_j + t u_j)/\delta\rfloor$ | `floor(pts / delta).long()` | (D·S, n) | the cubes of side $\delta$ met by every tube, sampled every $\delta/2$ along it |
| $N_\delta$ | `unique(einsum('mi,i->m', cells - low, radix))` | int | repeated $i$ ⇒ summed: one integer key per cube, so counting distinct cubes is one sort |
| $R_c u$ | `einsum('cij,nvj->cnvi', rot, poly)` | (3, N, 3, 2) | repeated $j$ ⇒ summed: three copies of the tree turned by 0°, 60°, 120° cover every direction |

Kakeya asked in 1917 for the smallest area in which a unit needle can be turned around. Besicovitch (1919, 1928) found that a set containing a unit segment in every direction can have measure zero, and Perron (1928) simplified the construction into the tree drawn here. Cut a triangle into $2^k$ slivers and slide them sideways. Every sliver keeps its directions, and after each round of Perron's step the overlap grows. The area obeys $|\bigcup T_i| \le (\alpha^{2k} + 2(1-\alpha))|T|$ (checked), and the best $\alpha$, found by trying 24 values at once, makes it fall like $\ln k / k$. Córdoba (1977) proved that the $\delta$-neighbourhood of a Kakeya set in the plane has area at least $c / \log(1/\delta)$, so this slow decay is unavoidable, and Davies (1971) proved that every planar Kakeya set has Hausdorff and Minkowski dimension 2. Three trees turned by 60° contain a unit needle in every direction (the red needle sweeps 180°); joining the positions with Pál's trick lets the needle actually rotate inside a set of area as small as you like. In $\mathbb R^n$ the Kakeya set conjecture says every such set has dimension $n$: tubes of radius $\delta$ in $\delta$-separated directions cannot overlap much, so their union has volume $\gtrsim_\varepsilon \delta^{\varepsilon}$. The right panels count $\delta$-cubes for two arrangements (all tubes through the origin, and random centres). The slope of $\log N$ against $\log(1/\delta)$ approaches $n$ in each case, as it must if the conjecture is true. A computer can only test particular arrangements, never all of them. Status (November 2026): $n = 2$ proved (Davies 1971). $n = 3$ proved by Hong Wang and Joshua Zahl (preprint 2025), after Wolff's bound $(n+2)/2$ (1995) and Katz–Łaba–Tao. For $n \ge 4$ the problem is open in the peer-reviewed literature. The AI-generated manuscript collection published by OpenAI (github.com/openai/math, families 074 and 079) claims the four-dimensional case and, through Sogge's local smoothing conjecture, every dimension. The repository says that some of its statements have been formalised in Lean, but these are unrefereed, machine-produced claims still awaiting checking by the mathematical community, so this page treats them as claimed, not proven. The conjecture matters well beyond geometry: it follows from the Fourier restriction and Bochner–Riesz conjectures of harmonic analysis.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/kakeya_conjecture/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    k_max = 6 if fast_mode() else params["k_max"]
    n_max = 3 if fast_mode() else params["n_max"]
    h = tensors.HALF_WIDTH

    # 1. Perron trees: 2^k thin triangles, shifted sideways, overlapping more with every round
    trees = [solve.perron_tree(k, dev) for k in range(k_max + 1)]
    print(f"{'k':>3} {'triangles':>9} {'alpha':>6} {'area / T':>9} {'Perron bound':>13} {'area × k':>9}")
    for t in trees:
        k, a = t["k"], t["alpha"]
        bound = a ** (2 * k) + 2 * (1 - a)
        print(f"{k:>3} {2**k:>9} {a:>6.3f} {t['area'] / h:>9.4f} {min(bound, 1):>13.4f} {t['area'] / h * k:>9.3f}")
    print("area shrinks with k, but only like ln k / k: Córdoba (1977) showed it cannot beat c / log N")

    # 2. the δ-neighbourhood of the Kakeya set made of three turned trees (all directions)
    kakeya = []
    for t in trees[3 : min(k_max, 9) + 1]:
        delta = 4 * h / 2 ** t["k"]  # two base widths: the tubes of this tree are this thin
        cells = solve.kakeya_cells(t, delta)
        kakeya.append((delta, cells))
        print(
            f"Kakeya set from three trees, k = {t['k']}: δ = {delta:.4f}, N(δ) = {cells:,}, "
            f"area ≈ {cells * delta**2:.3f}"
        )

    # 3. one δ-tube per direction in ℝⁿ: bush (all through 0) and random centres
    scales = {2: [1 / 8, 1 / 16, 1 / 32, 1 / 64, 1 / 128, 1 / 256], 3: [1 / 4, 1 / 8, 1 / 16, 1 / 32, 1 / 64]}
    scales[4] = [1 / 4, 1 / 8, 1 / 16]
    if fast_mode():
        scales = {2: scales[2][:4], 3: scales[3][:3]}
    tubes = {}
    for n in range(2, n_max + 1):
        for arrangement in ("bush", "random"):
            counts = solve.tube_counts(n, scales[n], arrangement, dev)
            slope = solve.dimension_slope(scales[n], counts)
            tubes[(n, arrangement)] = (scales[n], counts, slope)
            vols = ", ".join(f"{c * d**n:.3f}" for c, d in zip(counts, scales[n]))
            print(f"ℝ{n} {arrangement:>6}: N(δ)δⁿ = {vols};  log N / log(1/δ) slope ≈ {slope:.2f} (n = {n})")
    print("every arrangement tried has slope → n; the Kakeya conjecture says no arrangement can do better")

    u3 = tensors.directions(3, 0.25, dev)
    c3 = tensors.centres("random", len(u3), 3, 1, dev) * 0.6
    return plot.animate(trees, kakeya, tubes, (c3 - u3 / 2, c3 + u3 / 2), out_dir / "kakeya_conjecture.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/kakeya_conjecture/equation.py
r"""Mathematics → tensor translation.

1. Perron tree (Perron 1928, after Besicovitch 1919). Cut the triangle with apex (0, 1) and base [−h, h] on the x-axis
   into N = 2^k thin triangles T_i with bases [b_i, b_i + w₀], w₀ = 2h/N. Only horizontal shifts s_i are allowed, so
   every T_i keeps all its directions. At height y (0 ≤ y ≤ 1) triangle i covers the interval

        ℓ_i(y) = b_i (1 − y) + s_i,     r_i(y) = ℓ_i(y) + w₀ (1 − y),

   and the area of the union is ∫₀¹ |∪_i [ℓ_i(y), r_i(y)]| dy: a sort along i and a running maximum per row y.

   The shifts come from k rounds of Perron's step with a parameter α ∈ (½, 1): two neighbouring "hearts" (triangles
   with a common apex and adjacent bases [a, b], [b, c]) are made adjacent, then the right one is pushed left by
   d = (1 − α)(c − a). Their outer sides then cross at height α, so the new heart is the old pair scaled by α and the
   overlap above it is saved area. After k rounds the heart has height α^k and

        |∪ T_i| ≤ (α^{2k} + 2(1 − α)) |T|,

   which tends to 0 for α = 1 − ln k / k (so the area falls only like ln k / k = O(ln ln N / ln N)).

2. Tubes in ℝⁿ. A δ-tube is the δ-neighbourhood of a unit segment c + t u, |t| ≤ ½. Covering a union of tubes with
   cubes of side δ gives N(δ) cubes; its volume is ≈ N(δ) δⁿ and the Minkowski dimension is
   lim log N(δ) / log(1/δ). The Kakeya conjecture says that with one tube in every δ-separated direction this
   dimension is n: the union cannot be much smaller than a ball, |∪ tubes| ≳_ε δ^ε.
"""

import math

import torch


def perron_shifts(k: int, alpha: torch.Tensor, half_width: float) -> torch.Tensor:
    """Horizontal shifts s_i of the N = 2^k elementary triangles after k Perron rounds, for A values of α at once.

    alpha: (A,) → (A, N). Each round pairs the hearts (g, g+1): left heart base [L, L + w], right heart [R, R + w]."""
    n = 2**k
    a = alpha.double()[:, None]  # (A, 1)
    w = torch.full_like(a, 2 * half_width / n)  # heart base width (A, 1)
    left = -half_width + 2 * half_width * torch.arange(n, dtype=torch.float64, device=alpha.device) / n
    heart = left[None, :].repeat(len(alpha), 1)  # (A, G) base-left of every heart, G = N at the start
    shift = torch.zeros_like(heart)  # (A, N)
    for level in range(k):
        size = 2**level  # elementary triangles per heart before this round
        lhs, rhs = heart[:, 0::2], heart[:, 1::2]  # (A, G/2)
        delta = (lhs + w) - rhs - (1 - a) * 2 * w  # make adjacent, then push left by (1 − α)·(c − a)
        shift = shift.view(len(alpha), -1, 2, size)  # (A, G/2, 2, size): which half each triangle is in
        shift[:, :, 1, :] += delta[:, :, None]
        shift = shift.reshape(len(alpha), n)
        heart, w = lhs, 2 * a * w  # the new heart is the pair scaled by α
    return shift


def intervals(y: torch.Tensor, base_left: torch.Tensor, w0: float, shift: torch.Tensor) -> tuple:
    """ℓ_i(y) and r_i(y) for every row y and triangle i.   (Y,), (N,), (..., N) → (..., Y, N) twice"""
    lo = base_left[None, :] * (1 - y[:, None]) + shift[..., None, :]
    return lo, lo + w0 * (1 - y[:, None])


def union_length(lo: torch.Tensor, hi: torch.Tensor) -> torch.Tensor:
    """Length of ∪_i [lo_i, hi_i] along the last axis: sort by lo, then add what sticks out past the running max."""
    lo, order = torch.sort(lo, dim=-1)
    hi = torch.gather(hi, -1, order)
    reach = torch.cummax(hi, dim=-1).values  # furthest right end so far
    before = torch.cat([torch.full_like(reach[..., :1], -math.inf), reach[..., :-1]], dim=-1)
    return (hi - torch.maximum(lo, before)).clamp(min=0).sum(dim=-1)


def union_area(base_left: torch.Tensor, w0: float, shift: torch.Tensor, n_rows: int) -> torch.Tensor:
    """∫₀¹ |∪_i [ℓ_i(y), r_i(y)]| dy by the midpoint rule.   shift (..., N) → (...)"""
    y = (torch.arange(n_rows, dtype=torch.float64, device=shift.device) + 0.5) / n_rows
    lo, hi = intervals(y, base_left, w0, shift)
    return union_length(lo, hi).mean(dim=-1)


def inside_union(points: torch.Tensor, base_left: torch.Tensor, w0: float, shift: torch.Tensor) -> torch.Tensor:
    """Is each point (x, y), 0 ≤ y ≤ 1, in at least one shifted triangle?   (M, 2), (N,) → (M,) bool"""
    x, y = points[:, 0:1], points[:, 1:2]
    lo = base_left[None, :] * (1 - y) + shift[None, :]
    return ((x >= lo - 1e-12) & (x <= lo + w0 * (1 - y) + 1e-12)).any(dim=1)


def segment_cells(start: torch.Tensor, end: torch.Tensor, delta: float) -> torch.Tensor:
    """Indices of the δ-cubes met by every segment from start_j to end_j (sampled every δ/2 or finer).

    start, end: (J, n) → (J·S, n) int64 cube coordinates."""
    n_samples = int(math.ceil((end - start).norm(dim=1).max().item() / (delta / 2))) + 1
    t = torch.linspace(0, 1, n_samples, dtype=torch.float64, device=start.device)  # (S,)
    pts = start[:, None, :] + t[None, :, None] * (end - start)[:, None, :]  # (J, S, n)
    return torch.floor(pts / delta).long().reshape(-1, start.shape[1])


def count_cells(cells: torch.Tensor) -> int:
    """N(δ): number of distinct cubes (rows) in a (M, n) integer tensor, via one integer key per cube."""
    low = cells.min(dim=0).values
    span = int((cells.max(dim=0).values - low).max()) + 1
    radix = span ** torch.arange(cells.shape[1], device=cells.device)  # (n,) mixed-radix place values
    keys = torch.einsum("mi,i->m", cells - low, radix)  # repeated i ⇒ summed
    return len(torch.unique(keys))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/kakeya_conjecture/tensors.py
"""Build the concrete tensors: the thin triangles of the Perron tree, δ-separated directions and tube centres."""

import math

import torch

HALF_WIDTH = 1 / math.sqrt(3)  # base [−h, h] at distance 1 from the apex: a 60° triangle, all sides ≥ 1


def triangle_bases(k: int, device: torch.device) -> tuple[torch.Tensor, float]:
    """Left ends b_i of the N = 2^k bases [b_i, b_i + w₀] and their width w₀.   → (N,), float"""
    n = 2**k
    w0 = 2 * HALF_WIDTH / n
    return -HALF_WIDTH + w0 * torch.arange(n, dtype=torch.float64, device=device), w0


def directions(n: int, delta: float, device: torch.device) -> torch.Tensor:
    """About δ-separated unit vectors covering every line direction in ℝⁿ (one hemisphere).

    A grid of spacing δ on each face x_j = 1 of the cube [−1, 1]ⁿ, pushed out to the sphere: n·(2/δ + 1)^{n−1}
    directions, neighbours between δ/√n and δ apart.   → (D, n)"""
    g = torch.arange(-1, 1 + 1e-9, delta, dtype=torch.float64, device=device)  # (2/δ + 1,)
    face = torch.cartesian_prod(*([g] * (n - 1))).reshape(-1, n - 1)  # (F, n − 1)
    ones = torch.ones(len(face), 1, dtype=torch.float64, device=device)
    dirs = torch.cat([torch.cat([face[:, :j], ones, face[:, j:]], dim=1) for j in range(n)])  # (n·F, n)
    return dirs / dirs.norm(dim=1, keepdim=True)


def centres(arrangement: str, n_tubes: int, n: int, seed: int, device: torch.device) -> torch.Tensor:
    """Tube centres: all at the origin (a "bush": a Kakeya set shaped like a ball) or uniform in [−1, 1]ⁿ."""
    if arrangement == "bush":
        return torch.zeros(n_tubes, n, dtype=torch.float64, device=device)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    return (2 * torch.rand(n_tubes, n, generator=gen, dtype=torch.float64) - 1).to(device)


def rotation(angle: float) -> torch.Tensor:
    """2×2 rotation matrix."""
    c, s = math.cos(angle), math.sin(angle)
    return torch.tensor([[c, -s], [s, c]], dtype=torch.float64)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/kakeya_conjecture/solve.py
"""Build Perron trees with the best α, assemble a Kakeya set from three of them, and count δ-cubes of tube unions."""

import math

import torch

from . import tensors
from .equation import count_cells, perron_shifts, segment_cells, union_area


def perron_tree(k: int, device: torch.device, n_alpha: int = 24, n_rows: int = 2000) -> dict:
    """Perron tree with 2^k triangles; α chosen by trying n_alpha values at once (coarse rows), then a fine area."""
    base, w0 = tensors.triangle_bases(k, device)
    alphas = torch.linspace(0.5, 0.99, n_alpha, dtype=torch.float64, device=device)
    coarse = union_area(base, w0, perron_shifts(k, alphas, tensors.HALF_WIDTH), 200)  # (A,)
    alpha = alphas[coarse.argmin()]
    shift = perron_shifts(k, alpha[None], tensors.HALF_WIDTH)[0]  # (N,)
    area = union_area(base, w0, shift, n_rows).item()
    return {"k": k, "alpha": alpha.item(), "base": base, "w0": w0, "shift": shift, "area": area}


def tree_polygons(tree: dict) -> torch.Tensor:
    """Vertices of every shifted triangle: apex (s_i, 1), base corners (b_i + s_i, 0) and (b_i + w₀ + s_i, 0).

    → (N, 3, 2)"""
    s, b, w0 = tree["shift"], tree["base"], tree["w0"]
    one, zero = torch.ones_like(s), torch.zeros_like(s)
    return torch.stack(
        [torch.stack([s, one], 1), torch.stack([b + s, zero], 1), torch.stack([b + w0 + s, zero], 1)], dim=1
    )


def tree_centre(tree: dict) -> torch.Tensor:
    """Middle of the tree's bounding box, the point the three copies are turned about.   → (2,)"""
    x = tree_polygons(tree)[..., 0]
    return torch.stack([(x.min() + x.max()) / 2, torch.tensor(0.5, dtype=x.dtype, device=x.device)])


def kakeya_polygons(tree: dict) -> torch.Tensor:
    """Three copies of the tree turned by 0°, 60° and 120° about its centre: every line direction is covered.

    → (3N, 3, 2)"""
    poly = tree_polygons(tree) - tree_centre(tree)
    rot = torch.stack([tensors.rotation(a * math.pi / 3) for a in range(3)]).to(poly.device)  # (3, 2, 2)
    turned = torch.einsum("cij,nvj->cnvi", rot, poly)  # repeated j ⇒ summed: rotate every vertex of every copy
    return turned.reshape(-1, 3, 2)


def tree_segments(tree: dict, delta: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Segments from each apex to base points spaced ≤ δ/2: they sweep every triangle.   → (J, 2), (J, 2)"""
    m = max(2, int(math.ceil(tree["w0"] / (delta / 2))) + 1)
    frac = torch.linspace(0, 1, m, dtype=torch.float64, device=tree["shift"].device)
    s, b = tree["shift"][:, None], tree["base"][:, None]
    end_x = (b + tree["w0"] * frac[None, :] + s).reshape(-1)  # (N·m,)
    start = torch.stack([s.expand(-1, m).reshape(-1), torch.ones_like(end_x)], 1)
    return start, torch.stack([end_x, torch.zeros_like(end_x)], 1)


def kakeya_cells(tree: dict, delta: float) -> int:
    """N(δ) for the three-copy Kakeya set built from this tree."""
    start, end = tree_segments(tree, delta)
    shift = tree_centre(tree)
    rot = torch.stack([tensors.rotation(a * math.pi / 3) for a in range(3)]).to(start.device)
    s3 = torch.einsum("cij,mj->cmi", rot, start - shift).reshape(-1, 2)
    e3 = torch.einsum("cij,mj->cmi", rot, end - shift).reshape(-1, 2)
    return count_cells(segment_cells(s3, e3, delta))


def tube_counts(n: int, deltas: list[float], arrangement: str, device: torch.device, seed: int = 0) -> list[int]:
    """N(δ) for one unit tube in each δ-separated direction of ℝⁿ, centred by the given arrangement."""
    out = []
    for delta in deltas:
        u = tensors.directions(n, delta, device)  # (D, n)
        c = tensors.centres(arrangement, len(u), n, seed, device)  # (D, n)
        out.append(count_cells(segment_cells(c - u / 2, c + u / 2, delta)))
    return out


def dimension_slope(deltas: list[float], counts: list[int], last: int = 2) -> float:
    """Minkowski-dimension estimate: slope of log N(δ) against log(1/δ) over the last `last` + 1 scales."""
    x = torch.log(1 / torch.tensor(deltas[-last - 1 :], dtype=torch.float64))
    y = torch.log(torch.tensor(counts[-last - 1 :], dtype=torch.float64))
    x, y = x - x.mean(), y - y.mean()
    return (torch.einsum("i,i->", x, y) / torch.einsum("i,i->", x, x)).item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/kakeya_conjecture/plot.py
"""Perron trees refining, the three-tree Kakeya set with a needle in every direction, and δ-tube scaling in ℝⁿ."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import PolyCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import solve, tensors

N_FRAMES = 40


def animate(trees: list, kakeya: list, tubes: dict, segments3: tuple, path: Path) -> Path:
    h = tensors.HALF_WIDTH
    k_max = trees[-1]["k"]
    polys = [solve.tree_polygons(t).cpu().numpy() for t in trees]
    kpolys = [solve.kakeya_polygons(t).cpu().numpy() for t in trees]
    areas = np.array([t["area"] / h for t in trees])
    alphas = np.array([t["alpha"] for t in trees])
    ks = np.arange(k_max + 1)
    centres = [solve.tree_centre(t).cpu().numpy() for t in trees]

    fig = plt.figure(figsize=(12, 7.2))
    ax_t = fig.add_subplot(2, 3, 1)
    ax_k = fig.add_subplot(2, 3, 2)
    ax_a = fig.add_subplot(2, 3, 3)
    ax_3 = fig.add_subplot(2, 3, 4, projection="3d")
    ax_d = fig.add_subplot(2, 3, 5)
    ax_v = fig.add_subplot(2, 3, 6)

    tree_coll = PolyCollection(polys[0], facecolor="#60a5fa", edgecolor="#1e40af", lw=0.3)
    ax_t.add_collection(tree_coll)
    ax_t.plot([-h, 0, h, -h], [0, 1, 0, 0], color="#9ca3af", lw=1, ls="--")
    (needle_t,) = ax_t.plot([], [], color=ACCENT[1], lw=2.2)
    x_lo = min(p[..., 0].min() for p in polys) - 0.05
    x_hi = max(p[..., 0].max() for p in polys) + 0.05
    ax_t.set_xlim(x_lo, x_hi)
    ax_t.set_ylim(-0.05, 1.08)
    ax_t.set_aspect("equal")
    title_t = ax_t.set_title("", fontsize=9)

    kak_coll = PolyCollection(kpolys[0], facecolor="#60a5fa", edgecolor="#1e40af", lw=0.3)
    ax_k.add_collection(kak_coll)
    (needle_k,) = ax_k.plot([], [], color=ACCENT[1], lw=2.2)
    lim = max(np.abs(p).max() for p in kpolys) + 0.05
    ax_k.set_xlim(-lim, lim)
    ax_k.set_ylim(-lim, lim)
    ax_k.set_aspect("equal")
    title_k = ax_k.set_title("", fontsize=9)

    ax_a.plot(ks[1:], areas[1:], "o-", color=ACCENT[0], ms=4, label="Perron tree area / triangle")
    bound = np.minimum(alphas ** (2 * ks) + 2 * (1 - alphas), 1)
    ax_a.plot(ks[1:], bound[1:], "s--", color="#9ca3af", ms=3, label=r"bound $\alpha^{2k} + 2(1-\alpha)$")
    c = (areas[1:] * ks[1:]).min()
    ax_a.plot(ks[1:], c / ks[1:], ":", color=ACCENT[1], label=r"$c/k = c/\log_2 N$ (Córdoba shape)")
    (dot_a,) = ax_a.plot([], [], "o", color=ACCENT[1], ms=9, mfc="none", mew=2)
    ax_a.set_xlabel("refinement $k$ ($N = 2^k$ triangles)")
    ax_a.set_ylim(0, 1.05)
    ax_a.legend(fontsize=7)
    ax_a.set_title("area → 0, but only logarithmically", fontsize=9)

    start, end = (s.cpu().numpy() for s in segments3)
    ax_3.add_collection3d(Line3DCollection(np.stack([start, end], 1), colors=ACCENT[0], lw=0.7))
    for setter in (ax_3.set_xlim, ax_3.set_ylim, ax_3.set_zlim):
        setter(-1, 1)
    ax_3.set_xticklabels([])
    ax_3.set_yticklabels([])
    ax_3.set_zticklabels([])
    ax_3.set_title(rf"$\mathbb{{R}}^3$: one unit tube per direction ({len(start)} directions)", fontsize=9)

    styles = {2: ACCENT[0], 3: ACCENT[2], 4: ACCENT[3]}
    for (n, arr), (deltas, counts, slope) in tubes.items():
        inv = 1 / np.array(deltas)
        ls = "-" if arr == "bush" else "--"
        ax_d.loglog(inv, counts, ls, marker="o", ms=3, color=styles[n], label=f"n={n} {arr}: slope {slope:.2f}")
        vol = np.array(counts) * np.array(deltas) ** n
        ax_v.semilogx(inv, vol, ls, marker="o", ms=3, color=styles[n], label=f"n={n} {arr}")
    ax_d.set_xlabel(r"$1/\delta$")
    ax_d.set_ylabel(r"$N(\delta)$ cubes of side $\delta$")
    ax_d.legend(fontsize=6.5, loc="upper left")
    ax_d.set_title(r"Minkowski dimension: slope of $\log N$ vs $\log 1/\delta$", fontsize=9)

    kd = np.array([1 / d for d, _ in kakeya])
    kv = np.array([cells * d**2 for d, cells in kakeya])
    ax_v.semilogx(kd, kv, "-", marker="D", ms=4, color=ACCENT[1], lw=2, label="n=2 three Perron trees")
    ax_v.set_yscale("log")
    ax_v.set_xlabel(r"$1/\delta$")
    ax_v.set_ylabel(r"volume $\approx N(\delta)\,\delta^n$")
    ax_v.set_ylim(0.3, 300)
    ax_v.legend(fontsize=6.5, loc="upper right", ncol=2)
    ax_v.set_title("tube unions level off; the Perron set keeps shrinking", fontsize=9)

    hold = N_FRAMES // (k_max + 1)

    def needle(theta: float, centre: np.ndarray) -> tuple:
        d = 0.5 * np.array([math.cos(theta), math.sin(theta)])
        return [centre[0] - d[0], centre[0] + d[0]], [centre[1] - d[1], centre[1] + d[1]]

    def update(f: int):
        k = min(k_max, f // hold)
        tree_coll.set_verts(polys[k])
        kak_coll.set_verts(kpolys[k])
        edge = "#1e40af" if k <= 5 else "#60a5fa"  # thousands of dark edges would hide the slivers
        tree_coll.set_edgecolor(edge)
        kak_coll.set_edgecolor(edge)
        n_tri = 2**k
        # needle in the tree: direction (b, −1) inside the cone, along a segment of triangle i
        b = -h + 2 * h * ((f * 7) % N_FRAMES + 0.5) / N_FRAMES
        i = min(n_tri - 1, int((b + h) / (2 * h / n_tri)))
        s = trees[k]["shift"][i].item()
        apex, foot = np.array([s, 1.0]), np.array([b + s, 0.0])
        unit = (foot - apex) / np.linalg.norm(foot - apex)
        needle_t.set_data([apex[0], apex[0] + unit[0]], [apex[1], apex[1] + unit[1]])
        title_t.set_text(f"Perron tree: $2^{{{k}}}$ triangles, area {areas[k]:.1%} of the triangle")
        # needle in the Kakeya set: sweep all 180°, placed on the copy and triangle that hold that direction
        theta = math.pi * f / N_FRAMES
        phi = (theta + math.pi / 2) % math.pi  # copy c holds line angles c·60° − 90° + atan(b), |atan b| ≤ 30°
        copy = round(phi / (math.pi / 3)) % 3
        local = (phi - copy * math.pi / 3 + math.pi / 2) % math.pi - math.pi / 2
        b = math.tan(local)
        i = min(n_tri - 1, max(0, int((b + h) / (2 * h / n_tri))))
        s = trees[k]["shift"][i].item()
        rot = tensors.rotation(copy * math.pi / 3).numpy()
        a0 = rot @ (np.array([s, 1.0]) - centres[k])
        f0 = rot @ (np.array([b + s, 0.0]) - centres[k])
        unit = (f0 - a0) / np.linalg.norm(f0 - a0)
        needle_k.set_data([a0[0], a0[0] + unit[0]], [a0[1], a0[1] + unit[1]])
        title_k.set_text(
            f"three trees: a unit needle at {math.degrees(theta):3.0f}°, area ≤ {3 * trees[k]['area']:.3f}"
        )
        dot_a.set_data([k], [areas[k]] if k > 0 else [1.0])
        ax_3.view_init(elev=20, azim=30 + 300 * f / N_FRAMES)

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(N_FRAMES, N_FRAMES), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'k_max': 9,
    'n_max': 4
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.kakeya_conjecture")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.kakeya_conjecture.main")
problem_dir = Path("problems/maths/kakeya_conjecture")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
